In [ ]:
import anndata
import numpy as np
import pandas as pd
import gc

In [ ]:
hvg_genes = pd.read_csv('/jdata/qmy/VirtualCell/tahoe_100M/hvg_all_genes', header=None)[0].tolist()
hvg_set = set(hvg_genes)

In [ ]:
concs = [0.05, 0.5, 5.0]
conc_to_str = {0.05: '0.05', 0.5: '0.5', 5.0: '5.0'}
sampled_cells = {c: {} for c in concs}  # key: (drug, cell) -> list of expr arrays

In [ ]:
paths = [f"/jdata/qmy/VirtualCell/tahoe_100M/plate{i}_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad" for i in range(1, 15)]

for plate_idx, p in enumerate(paths):
    print(f"\nProcessing plate {plate_idx+1}/14: {p}") 
    ad = anndata.read_h5ad(p, backed='r')
    var_names = ad.var_names.tolist()
    hvg_indices = np.array([i for i, g in enumerate(var_names) if g in hvg_set], dtype=np.int32)
    cell_names = ad.obs['cell_name'].values
    drug_strings = ad.obs['drugname_drugconc'].values
    batch_size = 10000
    n_cells = len(cell_names)
    for start in range(0, n_cells, batch_size):
        end = min(start + batch_size, n_cells)
        expr_all = ad[start:end, :].X
        if hasattr(expr_all, 'toarray'):
            expr_all = expr_all.toarray()
        else:
            expr_all = np.asarray(expr_all)
        expr = expr_all[:, hvg_indices]
        
        # 处理每个细胞
        for j in range(end - start):
            cell_name = cell_names[start + j]
            drug_str = drug_strings[start + j]
            if not (isinstance(drug_str, str) and drug_str):
                continue
            try:
                drugs = eval(drug_str)
                expr_row = expr[j].copy() 
                
                for drug, conc, _ in drugs:
                    if conc not in conc_to_str:
                        continue
                    
                    key = (drug, cell_name)
                    conc_samples = sampled_cells[conc]
                    
                    if key in conc_samples:
                        if len(conc_samples[key]) < 1:
                            conc_samples[key].append(expr_row)
                    else:
                        conc_samples[key] = [expr_row]
                        
            except:
                continue
        
        del expr_all, expr
        gc.collect()
    
    ad.file.close()
    del ad
    gc.collect()
pseudo_bulk_results = {}

for conc in concs:
    conc_str = conc_to_str[conc]
    data = []
    index = []
    for key, expr_list in sampled_cells[conc].items():
        if expr_list:
            mean_expr = np.mean(expr_list, axis=0, dtype=np.float64)
            data.append(mean_expr)
            index.append(key)
    if data:
        pseudo_df = pd.DataFrame(
            np.array(data, dtype=np.float64),
            index=pd.MultiIndex.from_tuples(index, names=['drug', 'cell']),
            columns=hvg_genes
        )
        pseudo_bulk_results[conc_str] = pseudo_df
        output_file = f"/jdata/qmy/VirtualCell/group_mean_drug/sc_pseudo_bulk_10_{conc_str}uM.csv"
        pseudo_df.to_csv(output_file, float_format='%.10f')
    else:
        print(f"  No data for {conc}uM")